# GPT-2 Seinfeld Finetuning (LoRA)

Fine-tunes **GPT-2 medium** on Seinfeld scripts using **LoRA** (Low-Rank Adaptation) so only ~0.5% of parameters are trained — fits on a free Colab T4.

## What you need
- A Google account (for Colab + Drive)
- ~15 min on a free T4 GPU

## What it does
1. Clones the repo (contains scripts + preprocessed training data)
2. Installs dependencies
3. Mounts your Google Drive (checkpoint saved there so it survives session resets)
4. Trains for 10 epochs — loss should drop from ~3.5 → ~2.0
5. Saves a merged (LoRA-merged-into-base) checkpoint to Drive
6. Runs a quick inference sanity check

## Training data format
Each example in `train.jsonl` looks like:
```
{"text": "TOPIC: losing a parking spot\n\n[JERRY'S APARTMENT]\nJERRY: I had the spot! I had it!\nGEORGE: You had it, and then you lost it.\n[END]"}
```

## Before you start
**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Clone repo
!git clone https://github.com/detrin/gpt-seinfeld /content/gpt-seinfeld
%cd /content/gpt-seinfeld

In [ ]:
# Install dependencies (takes ~1 min)
!pip install -q "transformers>=4.44" "peft>=0.17.0" datasets accelerate torch

In [ ]:
# Mount Google Drive — you'll be prompted to sign in with your Google account.
# The trained checkpoint will be saved to My Drive/gpt2-seinfeld/
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Confirm you have a GPU. Should show Tesla T4 or better.
!nvidia-smi | head -10

In [ ]:
# Train (~10–15 min on T4).
# The merged checkpoint is saved to Google Drive when done.
import os

os.environ["CHECKPOINT_DIR"] = "/content/drive/MyDrive/gpt2-seinfeld"
os.environ["DATA_FILE"] = "data/processed/train.jsonl"

%run training/train.py

In [ ]:
# Quick inference test — try a few topics to judge output quality.
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
gen = pipeline("text-generation", model="/content/drive/MyDrive/gpt2-seinfeld", device=device)

for topic in ["losing a parking spot", "a bad haircut", "waiting for a table"]:
    result = gen(f"TOPIC: {topic}\n\n", max_new_tokens=200, do_sample=True, temperature=0.9)
    print(f"=== {topic} ===")
    print(result[0]["generated_text"])
    print()